# SimpleTMG - All Datasets + Smartbuilding

This Kaggle notebook runs the full SWT/FFT x original/dual attention experiment grid across the benchmark datasets plus the cleaned `smartbuilding/smart.csv` dataset.

Variants:
- `SWT_original`
- `SWT_dual`
- `FFT_original`
- `FFT_dual`

The notebook also syncs the current local repo changes into the Kaggle clone so the smartbuilding loader support and extended metrics are available during the run.


In [ ]:
!pip install PyWavelets gdown --quiet
import os
import json
import shutil
import subprocess
import pandas as pd
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
REPO_URL = "https://github.com/agamswami/SimpleTMG.git"
REPO_DIR = "/kaggle/working/SimpleTM"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!sed -i 's/np\.Inf/np.inf/g' utils/tools.py


In [ ]:
FILES_TO_SYNC = {"run.py": "import argparse\nimport torch\nfrom experiments.exp_long_term_forecasting import Exp_Long_Term_Forecast\nimport random\nimport numpy as np\nfrom model.SimpleTM import Model\nfrom model.SimpleTM_SWT import Model as Model_SWT\nfrom model.SimpleTM_FFT import Model as Model_FFT\n\nif __name__ == '__main__':\n\n    parser = argparse.ArgumentParser(description='iTransformer')\n\n    # basic config\n    parser.add_argument('--is_training', type=int, required=True, default=1, help='status')\n    parser.add_argument('--model_id', type=str, required=True, default='test', help='model id')\n\n    parser.add_argument('--model', type=str, required=True, default='SimpleTM',\n                        help='model name, options: [SimpleTM, SimpleTM_SWT, SimpleTM_FFT]')\n\n    # data loader\n    parser.add_argument('--data', type=str, required=True, default='custom', help='dataset type')\n    parser.add_argument('--root_path', type=str, default='./data/electricity/', help='root path of the data file')\n    parser.add_argument('--data_path', type=str, default='electricity.csv', help='data csv file')\n    parser.add_argument('--features', type=str, default='M',\n                        help='forecasting task, options:[M, S, MS]; M:multivariate predict multivariate, S:univariate predict univariate, MS:multivariate predict univariate')\n    parser.add_argument('--target', type=str, default='OT', help='target feature in S or MS task')\n    parser.add_argument('--freq', type=str, default='h',\n                        help='freq for time features encoding, options:[s:secondly, t:minutely, h:hourly, d:daily, b:business days, w:weekly, m:monthly], you can also use more detailed freq like 15min or 3h')\n    parser.add_argument('--checkpoints', type=str, default='./checkpoints/', help='location of model checkpoints')\n\n    # forecasting task\n    parser.add_argument('--seq_len', type=int, default=96, help='input sequence length')\n    parser.add_argument('--label_len', type=int, default=0, help='start token length')\n    parser.add_argument('--pred_len', type=int, default=96, help='prediction sequence length')\n\n    # model define\n    parser.add_argument('--enc_in', type=int, default=7, help='encoder input size')\n    parser.add_argument('--dec_in', type=int, default=7, help='decoder input size')\n    parser.add_argument('--c_out', type=int, default=7, help='output size') \n    parser.add_argument('--n_heads', type=int, default=8, help='num of heads')\n    parser.add_argument('--d_layers', type=int, default=1, help='num of decoder layers')\n    parser.add_argument('--moving_avg', type=int, default=25, help='window size of moving average')\n    parser.add_argument('--factor', type=int, default=1, help='attn factor')\n    parser.add_argument('--distil', action='store_false',\n                        help='whether to use distilling in encoder, using this argument means not using distilling',\n                        default=True)\n    parser.add_argument('--dropout', type=float, default=0.1, help='dropout')\n    parser.add_argument('--geomattn_dropout', type=float, default=0.5, help='dropout rate of the projection layer in the geometric attention')\n    parser.add_argument('--embed', type=str, default='timeF',\n                        help='time features encoding, options:[timeF, fixed, learned]')\n    parser.add_argument('--activation', type=str, default='gelu', help='activation')\n    parser.add_argument('--do_predict', action='store_true', help='whether to predict unseen future data')\n\n    # optimization\n    parser.add_argument('--num_workers', type=int, default=10, help='data loader num workers')\n    parser.add_argument('--itr', type=int, default=1, help='experiments times')\n    parser.add_argument('--train_epochs', type=int, default=10, help='train epochs')\n    parser.add_argument('--batch_size', type=int, default=32, help='batch size of train input data')\n    parser.add_argument('--patience', type=int, default=3, help='early stopping patience')\n    parser.add_argument('--learning_rate', type=float, default=0.0001, help='optimizer learning rate')\n    parser.add_argument('--des', type=str, default='test', help='exp description')\n    parser.add_argument('--loss', type=str, default='MSE', help='loss function')\n    parser.add_argument('--lradj', type=str, default='type1', help='adjust learning rate')\n    parser.add_argument('--pct_start', type=float, default=0.2, help='Warmup ratio for the learning rate scheduler')\n    parser.add_argument('--use_amp', action='store_true', help='use automatic mixed precision training', default=False)\n\n    # GPU\n    parser.add_argument('--use_gpu', type=bool, default=True, help='use gpu')\n    parser.add_argument('--gpu', type=int, default=0, help='gpu')\n    parser.add_argument('--use_multi_gpu', action='store_true', help='use multiple gpus', default=False)\n    parser.add_argument('--devices', type=str, default='0,1,2,3', help='device ids of multile gpus')\n\n    parser.add_argument('--exp_name', type=str, required=False, default='MTSF',\n                        help='experiemnt name, options:[MTSF, partial_train]')\n    parser.add_argument('--channel_independence', type=bool, default=False, help='whether to use channel_independence mechanism')\n    parser.add_argument('--inverse', action='store_true', help='inverse output data', default=False)\n    parser.add_argument('--class_strategy', type=str, default='projection', help='projection/average/cls_token')\n    parser.add_argument('--target_root_path', type=str, default='./data/electricity/', help='root path of the data file')\n    parser.add_argument('--target_data_path', type=str, default='electricity.csv', help='data file')\n    parser.add_argument('--efficient_training', type=bool, default=False, help='whether to use efficient_training (exp_name should be partial train)') # See Figure 8 of our paper for the detail\n    parser.add_argument('--use_norm', type=int, default=True, help='use norm and denorm')\n    parser.add_argument('--partial_start_index', type=int, default=0, help='the start index of variates for partial training, '\n                                                                           'you can select [partial_start_index, min(enc_in + partial_start_index, N)]')\n\n    # SimpleTM Arguments\n    parser.add_argument('--requires_grad', type=bool, default=True, help='Set to True to enable learnable wavelets')\n    parser.add_argument('--wv', type=str, default='db1', help='Wavelet filter type. Supports all wavelets available in PyTorch Wavelets')\n    parser.add_argument('--m', type=int, default=3, help='Number of levels for the stationary wavelet transform')\n    parser.add_argument('--kernel_size', default=None, help='Specify the length of randomly initialized wavelets (if not None)')\n    parser.add_argument('--alpha', type=float, default=1, help='Weight of the inner product score in geometric attention')\n    parser.add_argument('--l1_weight', type=float, default=5e-5, help='Weight of L1 loss')\n    parser.add_argument('--d_model', type=int, default=32, help='Dimensionality of pseudo tokens')\n    parser.add_argument('--d_ff', type=int, default=32, help='Dimensionality of the feedforward network')\n    parser.add_argument('--e_layers', type=int, default=1, help='Number of SimpleTM layers')\n    parser.add_argument('--compile', type=bool, default=False, help='Set to True to enable compilation, which can accelerate speed but may slightly impact performance')\n    parser.add_argument('--output_attention', action='store_true', help='Set to False to output attn, which can be used to compute training loss')\n    parser.add_argument('--attention_mode', type=str, default='original',\n                        help='attention variant, options: [original, dual]. dual adds a parallel temporal attention branch.')\n\n    parser.add_argument('--fix_seed', type=int, default=2025, help='gpu')\n    \n    args = parser.parse_args()\n    args.use_gpu = True if torch.cuda.is_available() and args.use_gpu else False\n\n    fix_seed = args.fix_seed\n    random.seed(fix_seed)\n    torch.manual_seed(fix_seed)\n    np.random.seed(fix_seed)\n\n    if args.use_gpu and args.use_multi_gpu:\n        args.devices = args.devices.replace(' ', '')\n        device_ids = args.devices.split(',')\n        args.device_ids = [int(id_) for id_ in device_ids]\n        args.gpu = args.device_ids[0]\n\n    print('Args in experiment:')\n    print(args)\n\n    Exp = Exp_Long_Term_Forecast\n\n    if args.is_training:\n        for ii in range(args.itr):\n            # setting record of experiments\n            setting = '{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}'.format(\n                args.model_id, \n                args.data,\n                args.seq_len,\n                args.pred_len,\n                args.d_model,\n                args.d_ff,\n                args.e_layers,   \n                args.wv,\n                args.kernel_size,\n                args.m,\n                args.alpha,\n                args.l1_weight, \n                args.learning_rate,\n                args.lradj,\n                args.batch_size,\n                args.fix_seed,\n                args.use_norm,\n                ii)\n\n            exp = Exp(args)  # set experiments\n            print('>>>>>>>start training : {}>>>>>>>>>>>>>>>>>>>>>>>>>>'.format(setting))\n            exp.train(setting)\n\n            print('>>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))\n            exp.test(setting)\n\n            if args.do_predict:\n                print('>>>>>>>predicting : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))\n                exp.predict(setting, True)\n\n            torch.cuda.empty_cache()\n    else:\n      \n        ii = 0\n        setting = '{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}'.format(\n            args.data,\n            args.seq_len,\n            args.pred_len,\n            args.d_model,\n            args.d_ff,\n            args.e_layers,\n            args.wv,\n            args.kernel_size,\n            args.m,\n            args.alpha,\n            args.l1_weight, \n            args.learning_rate,\n            args.lradj,\n            args.batch_size,\n            args.fix_seed,\n            args.use_norm,\n            ii)\n\n        exp = Exp(args)  # set experiments\n        print('>>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))\n        exp.test(setting, test=1)\n        torch.cuda.empty_cache()\n", "model/SimpleTM.py": "import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom layers.Transformer_Encoder import Encoder, EncoderLayer\nfrom layers.SWTAttention_Family import GeomAttentionLayer, GeomAttention\nfrom layers.ParallelAttention_Family import ParallelSWTGeomAttentionLayer\nfrom layers.Embed import DataEmbedding_inverted\n\n\nclass Model(nn.Module):\n    def __init__(self, configs):\n        super(Model, self).__init__()\n        self.seq_len = configs.seq_len\n        self.pred_len = configs.pred_len\n        self.output_attention = configs.output_attention\n        self.use_norm = configs.use_norm\n        self.geomattn_dropout = configs.geomattn_dropout\n        self.alpha = configs.alpha\n        self.kernel_size = configs.kernel_size\n        self.attention_mode = getattr(configs, 'attention_mode', 'original')\n\n        attention_layer_cls = GeomAttentionLayer\n        if self.attention_mode == 'dual':\n            attention_layer_cls = ParallelSWTGeomAttentionLayer\n\n        enc_embedding = DataEmbedding_inverted(configs.seq_len, configs.d_model, \n                                               configs.embed, configs.freq, configs.dropout)\n        self.enc_embedding = enc_embedding\n\n        encoder = Encoder(\n            [  \n                EncoderLayer(\n                    attention_layer_cls(\n                        GeomAttention(\n                            False, configs.factor, attention_dropout=configs.dropout, \n                            output_attention=configs.output_attention, alpha=self.alpha\n                        ),\n                        configs.d_model, \n                        requires_grad=configs.requires_grad, \n                        wv=configs.wv, \n                        m=configs.m, \n                        d_channel=configs.dec_in, \n                        kernel_size=self.kernel_size, \n                        geomattn_dropout=self.geomattn_dropout\n                    ),\n                    configs.d_model,\n                    configs.d_ff,\n                    dropout=configs.dropout,\n                    activation=configs.activation,\n                ) for l in range(configs.e_layers) \n            ],\n            norm_layer=torch.nn.LayerNorm(configs.d_model)\n        )\n        self.encoder = encoder\n\n        projector = nn.Linear(configs.d_model, self.pred_len, bias=True)\n        self.projector = projector\n\n\n    def forecast(self, x_enc, x_mark_enc, x_dec, x_mark_dec):\n        if self.use_norm:\n            means = x_enc.mean(1, keepdim=True).detach()\n            x_enc = x_enc - means\n            stdev = torch.sqrt(torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)\n            # x_enc /= stdev\n            x_enc = x_enc / stdev\n\n        _, _, N = x_enc.shape\n\n        enc_embedding = self.enc_embedding\n        encoder = self.encoder\n        projector = self.projector\n        # Linear Projection             B L N -> B L' (pseudo temporal tokens) N \n        enc_out = enc_embedding(x_enc, x_mark_enc) \n\n        # SimpleTM Layer                B L' N -> B L' N \n        enc_out, attns = encoder(enc_out, attn_mask=None)\n\n        # Output Projection             B L' N -> B H (Horizon) N\n        dec_out = projector(enc_out).permute(0, 2, 1)[:, :, :N] \n\n        if self.use_norm:\n            dec_out = dec_out * (stdev[:, 0, :].unsqueeze(1).repeat(1, self.pred_len, 1))\n            dec_out = dec_out + (means[:, 0, :].unsqueeze(1).repeat(1, self.pred_len, 1))\n\n        return dec_out, attns\n\n\n    def forward(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask=None):\n        dec_out, attns = self.forecast(x_enc, None, None, None)\n        return dec_out, attns \n", "model/SimpleTM_SWT.py": "\"\"\"\nSimpleTM_SWT: SWT-Only Tokenization Model\n\nThis is the original SimpleTM model documented as the SWT-only tokenization baseline.\nIt uses Stationary Wavelet Transform (SWT) for tokenization and Geometric Attention\n(wedge product + dot product) for inter-variate attention.\n\nPipeline:\n    1. Instance Normalization (RevIN)\n    2. Inverted Embedding: (B, L, N) \u2192 (B, N, d_model) \u2014 channel-independent temporal tokens\n    3. SWT Tokenization: Decomposes each token via SWT into multi-scale coefficients\n    4. Geometric Attention: Wedge product scoring across variates at each scale\n    5. SWT Reconstruction: Inverse SWT to reconstruct from multi-scale representation\n    6. Output Projection: (B, N, d_model) \u2192 (B, H, N) \u2014 forecast horizon\n    7. De-normalization\n\"\"\"\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom layers.Transformer_Encoder import Encoder, EncoderLayer\nfrom layers.SWTAttention_Family import GeomAttentionLayer, GeomAttention\nfrom layers.ParallelAttention_Family import ParallelSWTGeomAttentionLayer\nfrom layers.Embed import DataEmbedding_inverted\n\n\nclass Model(nn.Module):\n    \"\"\"\n    SimpleTM with SWT-Only Tokenization (Original Baseline).\n    \n    This is functionally identical to the original SimpleTM model.\n    Kept as a separate file for clean ablation study comparison.\n    \n    Args:\n        configs: Configuration namespace containing:\n            - seq_len (int): Input sequence length\n            - pred_len (int): Prediction horizon length\n            - output_attention (bool): Whether to output attention weights\n            - use_norm (bool): Whether to apply instance normalization\n            - geomattn_dropout (float): Dropout rate in attention projections\n            - alpha (float): Weight balancing dot product vs wedge product in GeomAttention\n            - kernel_size (int|None): Wavelet kernel size (None = use wavelet default)\n            - d_model (int): Pseudo token dimensionality\n            - d_ff (int): Feed-forward network dimensionality\n            - e_layers (int): Number of encoder layers\n            - enc_in (int): Number of input variates\n            - dec_in (int): Number of channels for wavelet embedding\n            - embed (str): Time embedding type\n            - freq (str): Data frequency\n            - dropout (float): General dropout\n            - requires_grad (bool): Whether wavelet filters are learnable\n            - wv (str): Wavelet family (e.g., 'db1', 'db2')\n            - m (int): Number of SWT decomposition levels\n            - factor (int): Attention factor\n            - activation (str): Activation function ('relu' or 'gelu')\n    \"\"\"\n    def __init__(self, configs):\n        super(Model, self).__init__()\n        self.seq_len = configs.seq_len\n        self.pred_len = configs.pred_len\n        self.output_attention = configs.output_attention\n        self.use_norm = configs.use_norm\n        self.geomattn_dropout = configs.geomattn_dropout\n        self.alpha = configs.alpha\n        self.kernel_size = configs.kernel_size\n        self.attention_mode = getattr(configs, 'attention_mode', 'original')\n\n        attention_layer_cls = GeomAttentionLayer\n        if self.attention_mode == 'dual':\n            attention_layer_cls = ParallelSWTGeomAttentionLayer\n\n        # Step 1: Inverted Embedding \u2014 project temporal dim to d_model\n        enc_embedding = DataEmbedding_inverted(configs.seq_len, configs.d_model, \n                                               configs.embed, configs.freq, configs.dropout)\n        self.enc_embedding = enc_embedding\n\n        # Step 2: Encoder with SWT tokenization + Geometric Attention\n        encoder = Encoder(\n            [  \n                EncoderLayer(\n                    attention_layer_cls(\n                        GeomAttention(\n                            False, configs.factor, attention_dropout=configs.dropout, \n                            output_attention=configs.output_attention, alpha=self.alpha\n                        ),\n                        configs.d_model, \n                        requires_grad=configs.requires_grad, \n                        wv=configs.wv, \n                        m=configs.m, \n                        d_channel=configs.dec_in, \n                        kernel_size=self.kernel_size, \n                        geomattn_dropout=self.geomattn_dropout\n                    ),\n                    configs.d_model,\n                    configs.d_ff,\n                    dropout=configs.dropout,\n                    activation=configs.activation,\n                ) for l in range(configs.e_layers) \n            ],\n            norm_layer=torch.nn.LayerNorm(configs.d_model)\n        )\n        self.encoder = encoder\n\n        # Step 3: Output Projection \u2014 project d_model to prediction horizon\n        projector = nn.Linear(configs.d_model, self.pred_len, bias=True)\n        self.projector = projector\n\n\n    def forecast(self, x_enc, x_mark_enc, x_dec, x_mark_dec):\n        \"\"\"\n        Forward pass for forecasting.\n        \n        Args:\n            x_enc: (B, L, N) input time series\n            x_mark_enc: Time features (unused, passed as None)\n            x_dec: Decoder input (unused)\n            x_mark_dec: Decoder time features (unused)\n            \n        Returns:\n            dec_out: (B, H, N) forecasted values\n            attns: List of attention weights from each encoder layer\n        \"\"\"\n        # Instance Normalization (RevIN-style)\n        if self.use_norm:\n            means = x_enc.mean(1, keepdim=True).detach()\n            x_enc = x_enc - means\n            stdev = torch.sqrt(torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)\n            x_enc = x_enc / stdev\n\n        _, _, N = x_enc.shape\n\n        enc_embedding = self.enc_embedding\n        encoder = self.encoder\n        projector = self.projector\n\n        # Inverted Embedding:       B L N -> B N d_model (pseudo temporal tokens)\n        enc_out = enc_embedding(x_enc, x_mark_enc) \n\n        # Encoder (SWT + GeomAttn): B N d_model -> B N d_model\n        enc_out, attns = encoder(enc_out, attn_mask=None)\n\n        # Output Projection:        B N d_model -> B H N\n        dec_out = projector(enc_out).permute(0, 2, 1)[:, :, :N] \n\n        # De-normalization\n        if self.use_norm:\n            dec_out = dec_out * (stdev[:, 0, :].unsqueeze(1).repeat(1, self.pred_len, 1))\n            dec_out = dec_out + (means[:, 0, :].unsqueeze(1).repeat(1, self.pred_len, 1))\n\n        return dec_out, attns\n\n\n    def forward(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask=None):\n        \"\"\"\n        Main forward method.\n        \n        Args:\n            x_enc: (B, L, N) input time series\n            x_mark_enc: Time features\n            x_dec: Decoder input\n            x_mark_dec: Decoder time features\n            mask: Optional mask\n            \n        Returns:\n            dec_out: (B, H, N) forecasted values\n            attns: Attention weights\n        \"\"\"\n        dec_out, attns = self.forecast(x_enc, None, None, None)\n        return dec_out, attns \n", "model/SimpleTM_FFT.py": "\"\"\"\nSimpleTM_FFT: FFT-Only Tokenization Model\n\nThis variant replaces the Stationary Wavelet Transform (SWT) tokenization \nwith Fast Fourier Transform (FFT) spectral decomposition, while keeping \nthe Geometric Attention (wedge product + dot product) mechanism unchanged.\n\nInstead of decomposing signals into wavelet approximation and detail \ncoefficients, FFT tokenization splits the frequency spectrum into m+1 \nbands, creating spectral tokens at different frequency scales.\n\nPipeline:\n    1. Instance Normalization (RevIN)\n    2. Inverted Embedding: (B, L, N) \u2192 (B, N, d_model) \u2014 channel-independent temporal tokens\n    3. FFT Tokenization: Decomposes each token via FFT into frequency band signals\n    4. Geometric Attention: Wedge product scoring across variates at each frequency band\n    5. FFT Reconstruction: Sum frequency bands to reconstruct signal\n    6. Output Projection: (B, N, d_model) \u2192 (B, H, N) \u2014 forecast horizon\n    7. De-normalization\n\"\"\"\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom layers.Transformer_Encoder import Encoder, EncoderLayer\nfrom layers.FFTAttention_Family import FFTGeomAttentionLayer\nfrom layers.ParallelAttention_Family import ParallelFFTGeomAttentionLayer\nfrom layers.SWTAttention_Family import GeomAttention\nfrom layers.Embed import DataEmbedding_inverted\n\n\nclass Model(nn.Module):\n    \"\"\"\n    SimpleTM with FFT-Only Tokenization.\n    \n    Replaces SWT wavelet decomposition with FFT spectral band decomposition.\n    Geometric attention (wedge product scoring) remains unchanged.\n    \n    Args:\n        configs: Configuration namespace containing:\n            - seq_len (int): Input sequence length\n            - pred_len (int): Prediction horizon length\n            - output_attention (bool): Whether to output attention weights\n            - use_norm (bool): Whether to apply instance normalization\n            - geomattn_dropout (float): Dropout rate in attention projections\n            - alpha (float): Weight balancing dot product vs wedge product\n            - d_model (int): Pseudo token dimensionality\n            - d_ff (int): Feed-forward network dimensionality\n            - e_layers (int): Number of encoder layers\n            - enc_in (int): Number of input variates\n            - dec_in (int): Number of channels for FFT embedding\n            - embed (str): Time embedding type\n            - freq (str): Data frequency\n            - dropout (float): General dropout\n            - m (int): Number of FFT frequency bands (m+1 total bands)\n            - factor (int): Attention factor\n            - activation (str): Activation function ('relu' or 'gelu')\n    \"\"\"\n    def __init__(self, configs):\n        super(Model, self).__init__()\n        self.seq_len = configs.seq_len\n        self.pred_len = configs.pred_len\n        self.output_attention = configs.output_attention\n        self.use_norm = configs.use_norm\n        self.geomattn_dropout = configs.geomattn_dropout\n        self.alpha = configs.alpha\n        self.attention_mode = getattr(configs, 'attention_mode', 'original')\n\n        attention_layer_cls = FFTGeomAttentionLayer\n        if self.attention_mode == 'dual':\n            attention_layer_cls = ParallelFFTGeomAttentionLayer\n\n        # Step 1: Inverted Embedding \u2014 project temporal dim to d_model\n        enc_embedding = DataEmbedding_inverted(configs.seq_len, configs.d_model, \n                                               configs.embed, configs.freq, configs.dropout)\n        self.enc_embedding = enc_embedding\n\n        # Step 2: Encoder with FFT tokenization + Geometric Attention\n        encoder = Encoder(\n            [  \n                EncoderLayer(\n                    attention_layer_cls(\n                        GeomAttention(\n                            False, configs.factor, attention_dropout=configs.dropout, \n                            output_attention=configs.output_attention, alpha=self.alpha\n                        ),\n                        configs.d_model, \n                        m=configs.m, \n                        d_channel=configs.dec_in, \n                        geomattn_dropout=self.geomattn_dropout\n                    ),\n                    configs.d_model,\n                    configs.d_ff,\n                    dropout=configs.dropout,\n                    activation=configs.activation,\n                ) for l in range(configs.e_layers) \n            ],\n            norm_layer=torch.nn.LayerNorm(configs.d_model)\n        )\n        self.encoder = encoder\n\n        # Step 3: Output Projection \u2014 project d_model to prediction horizon\n        projector = nn.Linear(configs.d_model, self.pred_len, bias=True)\n        self.projector = projector\n\n\n    def forecast(self, x_enc, x_mark_enc, x_dec, x_mark_dec):\n        \"\"\"\n        Forward pass for forecasting.\n        \n        Args:\n            x_enc: (B, L, N) input time series\n            x_mark_enc: Time features (unused, passed as None)\n            x_dec: Decoder input (unused)\n            x_mark_dec: Decoder time features (unused)\n            \n        Returns:\n            dec_out: (B, H, N) forecasted values\n            attns: List of attention weights from each encoder layer\n        \"\"\"\n        # Instance Normalization (RevIN-style)\n        if self.use_norm:\n            means = x_enc.mean(1, keepdim=True).detach()\n            x_enc = x_enc - means\n            stdev = torch.sqrt(torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)\n            x_enc = x_enc / stdev\n\n        _, _, N = x_enc.shape\n\n        enc_embedding = self.enc_embedding\n        encoder = self.encoder\n        projector = self.projector\n\n        # Inverted Embedding:       B L N -> B N d_model (pseudo temporal tokens)\n        enc_out = enc_embedding(x_enc, x_mark_enc) \n\n        # Encoder (FFT + GeomAttn): B N d_model -> B N d_model\n        enc_out, attns = encoder(enc_out, attn_mask=None)\n\n        # Output Projection:        B N d_model -> B H N\n        dec_out = projector(enc_out).permute(0, 2, 1)[:, :, :N] \n\n        # De-normalization\n        if self.use_norm:\n            dec_out = dec_out * (stdev[:, 0, :].unsqueeze(1).repeat(1, self.pred_len, 1))\n            dec_out = dec_out + (means[:, 0, :].unsqueeze(1).repeat(1, self.pred_len, 1))\n\n        return dec_out, attns\n\n\n    def forward(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask=None):\n        \"\"\"\n        Main forward method.\n        \n        Args:\n            x_enc: (B, L, N) input time series\n            x_mark_enc: Time features\n            x_dec: Decoder input\n            x_mark_dec: Decoder time features\n            mask: Optional mask\n            \n        Returns:\n            dec_out: (B, H, N) forecasted values\n            attns: Attention weights\n        \"\"\"\n        dec_out, attns = self.forecast(x_enc, None, None, None)\n        return dec_out, attns \n", "layers/ParallelAttention_Family.py": "import torch\nimport torch.nn as nn\nfrom math import sqrt\n\nfrom layers.SWTAttention_Family import GeomAttentionLayer\nfrom layers.FFTAttention_Family import FFTGeomAttentionLayer\n\n\nclass TemporalAxisAttention(nn.Module):\n    \"\"\"Self-attention over the latent time axis after inverted embedding.\"\"\"\n\n    def __init__(self, d_channel, attention_dropout=0.1):\n        super().__init__()\n        self.scale = 1.0 / sqrt(d_channel)\n        self.query_projection = nn.Linear(d_channel, d_channel)\n        self.key_projection = nn.Linear(d_channel, d_channel)\n        self.value_projection = nn.Linear(d_channel, d_channel)\n        self.out_projection = nn.Linear(d_channel, d_channel)\n        self.dropout = nn.Dropout(attention_dropout)\n\n    def forward(self, queries, keys, values):\n        # Convert (B, N, D) into time-major tokens (B, D, N).\n        queries = queries.transpose(1, 2)\n        keys = keys.transpose(1, 2)\n        values = values.transpose(1, 2)\n\n        q = self.query_projection(queries)\n        k = self.key_projection(keys)\n        v = self.value_projection(values)\n\n        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale\n        attn = self.dropout(torch.softmax(scores, dim=-1))\n        out = torch.matmul(attn, v)\n        out = self.out_projection(out)\n\n        return out.transpose(1, 2), scores.abs().mean()\n\n\nclass _ParallelAttentionBase(nn.Module):\n    def __init__(self, original_attention_layer, d_model, d_channel, branch_dropout=0.1):\n        super().__init__()\n        self.original_attention_layer = original_attention_layer\n        self.temporal_attention = TemporalAxisAttention(\n            d_channel=d_channel, attention_dropout=branch_dropout\n        )\n        self.gate = nn.Sequential(\n            nn.Linear(2 * d_model, d_model),\n            nn.GELU(),\n            nn.Linear(d_model, 1),\n        )\n\n    def forward(self, queries, keys, values, attn_mask=None, tau=None, delta=None):\n        channel_out, channel_attn = self.original_attention_layer(\n            queries, keys, values, attn_mask=attn_mask, tau=tau, delta=delta\n        )\n        temporal_out, temporal_attn = self.temporal_attention(queries, keys, values)\n\n        pooled = torch.cat(\n            [channel_out.mean(dim=1), temporal_out.mean(dim=1)], dim=-1\n        )\n        gate = torch.sigmoid(self.gate(pooled)).unsqueeze(-1)\n        out = gate * channel_out + (1.0 - gate) * temporal_out\n\n        return out, 0.5 * (channel_attn + temporal_attn)\n\n\nclass ParallelSWTGeomAttentionLayer(_ParallelAttentionBase):\n    \"\"\"Original SWT geometric attention plus a parallel temporal branch.\"\"\"\n\n    def __init__(\n        self,\n        attention,\n        d_model,\n        requires_grad=True,\n        wv=\"db2\",\n        m=2,\n        kernel_size=None,\n        d_channel=None,\n        geomattn_dropout=0.5,\n    ):\n        original_attention_layer = GeomAttentionLayer(\n            attention,\n            d_model,\n            requires_grad=requires_grad,\n            wv=wv,\n            m=m,\n            kernel_size=kernel_size,\n            d_channel=d_channel,\n            geomattn_dropout=geomattn_dropout,\n        )\n        super().__init__(\n            original_attention_layer=original_attention_layer,\n            d_model=d_model,\n            d_channel=d_channel,\n            branch_dropout=geomattn_dropout,\n        )\n\n\nclass ParallelFFTGeomAttentionLayer(_ParallelAttentionBase):\n    \"\"\"Original FFT geometric attention plus a parallel temporal branch.\"\"\"\n\n    def __init__(self, attention, d_model, m=2, d_channel=None, geomattn_dropout=0.5):\n        original_attention_layer = FFTGeomAttentionLayer(\n            attention,\n            d_model,\n            m=m,\n            d_channel=d_channel,\n            geomattn_dropout=geomattn_dropout,\n        )\n        super().__init__(\n            original_attention_layer=original_attention_layer,\n            d_model=d_model,\n            d_channel=d_channel,\n            branch_dropout=geomattn_dropout,\n        )\n", "data_provider/data_loader.py": "import os\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\nfrom sklearn.preprocessing import StandardScaler\nfrom utils.timefeatures import time_features\nimport warnings\n\nwarnings.filterwarnings('ignore')\n\n\ndef _resolve_datetime_column(df_raw):\n    for name in df_raw.columns:\n        if name.lower() in {'date', 'datetime', 'timestamp', 'time'}:\n            return name\n    raise ValueError(\n        \"No datetime column found. Expected one of: date, datetime, timestamp, time.\"\n    )\n\n\ndef _resolve_target_column(df_raw, target, features):\n    if target in df_raw.columns:\n        return target\n\n    lower_to_actual = {col.lower(): col for col in df_raw.columns}\n    if target.lower() in lower_to_actual:\n        return lower_to_actual[target.lower()]\n\n    if features == 'M':\n        return None\n\n    raise ValueError(\n        f\"Target column '{target}' not found in dataset columns: {list(df_raw.columns)}\"\n    )\n\n\ndef _prepare_custom_dataframe(df_raw, target, features):\n    date_col = _resolve_datetime_column(df_raw)\n    target_col = _resolve_target_column(df_raw, target, features)\n\n    feature_cols = [col for col in df_raw.columns if col != date_col]\n    if target_col is not None and target_col in feature_cols:\n        feature_cols.remove(target_col)\n\n    ordered_cols = [date_col] + feature_cols\n    rename_map = {date_col: 'date'}\n\n    if target_col is not None:\n        ordered_cols.append(target_col)\n        rename_map[target_col] = target\n\n    df_prepared = df_raw[ordered_cols].rename(columns=rename_map)\n    return df_prepared\n\nclass Dataset_ETT_hour(Dataset):\n    def __init__(self, root_path, flag='train', size=None,\n                 features='S', data_path='ETTh1.csv',\n                 target='OT', scale=True, timeenc=0, freq='h'):\n        if size == None:\n            self.seq_len = 24 * 4 * 4\n            self.label_len = 24 * 4\n            self.pred_len = 24 * 4\n        else:\n            self.seq_len = size[0]\n            self.label_len = size[1]\n            self.pred_len = size[2]\n        assert flag in ['train', 'test', 'val']\n        type_map = {'train': 0, 'val': 1, 'test': 2}\n        self.set_type = type_map[flag]\n\n        self.features = features\n        self.target = target\n        self.scale = scale\n        self.timeenc = timeenc\n        self.freq = freq\n\n        self.root_path = root_path\n        self.data_path = data_path\n        self.__read_data__()\n\n    def __read_data__(self):\n        self.scaler = StandardScaler()\n        df_raw = pd.read_csv(os.path.join(self.root_path,\n                                          self.data_path))\n\n        border1s = [0, 12 * 30 * 24 - self.seq_len, 12 * 30 * 24 + 4 * 30 * 24 - self.seq_len]\n        border2s = [12 * 30 * 24, 12 * 30 * 24 + 4 * 30 * 24, 12 * 30 * 24 + 8 * 30 * 24]\n        border1 = border1s[self.set_type]\n        border2 = border2s[self.set_type]\n\n        if self.features == 'M' or self.features == 'MS':\n            cols_data = df_raw.columns[1:]\n            df_data = df_raw[cols_data]\n        elif self.features == 'S':\n            df_data = df_raw[[self.target]]\n\n        if self.scale:\n            train_data = df_data[border1s[0]:border2s[0]]\n            self.scaler.fit(train_data.values)\n            data = self.scaler.transform(df_data.values)\n        else:\n            data = df_data.values\n\n        df_stamp = df_raw[['date']][border1:border2]\n        df_stamp['date'] = pd.to_datetime(df_stamp.date)\n        if self.timeenc == 0:\n            df_stamp['month'] = df_stamp.date.apply(lambda row: row.month, 1)\n            df_stamp['day'] = df_stamp.date.apply(lambda row: row.day, 1)\n            df_stamp['weekday'] = df_stamp.date.apply(lambda row: row.weekday(), 1)\n            df_stamp['hour'] = df_stamp.date.apply(lambda row: row.hour, 1)\n            data_stamp = df_stamp.drop(['date'], 1).values\n        elif self.timeenc == 1:\n            data_stamp = time_features(pd.to_datetime(df_stamp['date'].values), freq=self.freq)\n            data_stamp = data_stamp.transpose(1, 0)\n\n        self.data_x = data[border1:border2]\n        self.data_y = data[border1:border2]\n        self.data_stamp = data_stamp\n\n    def __getitem__(self, index):\n        s_begin = index\n        s_end = s_begin + self.seq_len\n        r_begin = s_end - self.label_len\n        r_end = r_begin + self.label_len + self.pred_len\n\n        seq_x = self.data_x[s_begin:s_end]\n        seq_y = self.data_y[r_begin:r_end]\n        seq_x_mark = self.data_stamp[s_begin:s_end]\n        seq_y_mark = self.data_stamp[r_begin:r_end]\n\n        return seq_x, seq_y, seq_x_mark, seq_y_mark\n\n    def __len__(self):\n        return len(self.data_x) - self.seq_len - self.pred_len + 1\n\n    def inverse_transform(self, data):\n        return self.scaler.inverse_transform(data)\n\n\nclass Dataset_ETT_minute(Dataset):\n    def __init__(self, root_path, flag='train', size=None,\n                 features='S', data_path='ETTm1.csv',\n                 target='OT', scale=True, timeenc=0, freq='t'):\n        if size == None:\n            self.seq_len = 24 * 4 * 4\n            self.label_len = 24 * 4\n            self.pred_len = 24 * 4\n        else:\n            self.seq_len = size[0]\n            self.label_len = size[1]\n            self.pred_len = size[2]\n        assert flag in ['train', 'test', 'val']\n        type_map = {'train': 0, 'val': 1, 'test': 2}\n        self.set_type = type_map[flag]\n\n        self.features = features\n        self.target = target\n        self.scale = scale\n        self.timeenc = timeenc\n        self.freq = freq\n\n        self.root_path = root_path\n        self.data_path = data_path\n        self.__read_data__()\n\n    def __read_data__(self):\n        self.scaler = StandardScaler()\n        df_raw = pd.read_csv(os.path.join(self.root_path,\n                                          self.data_path))\n\n        border1s = [0, 12 * 30 * 24 * 4 - self.seq_len, 12 * 30 * 24 * 4 + 4 * 30 * 24 * 4 - self.seq_len]\n        border2s = [12 * 30 * 24 * 4, 12 * 30 * 24 * 4 + 4 * 30 * 24 * 4, 12 * 30 * 24 * 4 + 8 * 30 * 24 * 4]\n        border1 = border1s[self.set_type]\n        border2 = border2s[self.set_type]\n\n        if self.features == 'M' or self.features == 'MS':\n            cols_data = df_raw.columns[1:]\n            df_data = df_raw[cols_data]\n        elif self.features == 'S':\n            df_data = df_raw[[self.target]]\n\n        if self.scale:\n            train_data = df_data[border1s[0]:border2s[0]]\n            self.scaler.fit(train_data.values)\n            data = self.scaler.transform(df_data.values)\n        else:\n            data = df_data.values\n\n        df_stamp = df_raw[['date']][border1:border2]\n        df_stamp['date'] = pd.to_datetime(df_stamp.date)\n        if self.timeenc == 0:\n            df_stamp['month'] = df_stamp.date.apply(lambda row: row.month, 1)\n            df_stamp['day'] = df_stamp.date.apply(lambda row: row.day, 1)\n            df_stamp['weekday'] = df_stamp.date.apply(lambda row: row.weekday(), 1)\n            df_stamp['hour'] = df_stamp.date.apply(lambda row: row.hour, 1)\n            df_stamp['minute'] = df_stamp.date.apply(lambda row: row.minute, 1)\n            df_stamp['minute'] = df_stamp.minute.map(lambda x: x // 15)\n            data_stamp = df_stamp.drop(['date'], 1).values\n        elif self.timeenc == 1:\n            data_stamp = time_features(pd.to_datetime(df_stamp['date'].values), freq=self.freq)\n            data_stamp = data_stamp.transpose(1, 0)\n\n        self.data_x = data[border1:border2]\n        self.data_y = data[border1:border2]\n        self.data_stamp = data_stamp\n\n    def __getitem__(self, index):\n        s_begin = index\n        s_end = s_begin + self.seq_len\n        r_begin = s_end - self.label_len\n        r_end = r_begin + self.label_len + self.pred_len\n\n        seq_x = self.data_x[s_begin:s_end]\n        seq_y = self.data_y[r_begin:r_end]\n        seq_x_mark = self.data_stamp[s_begin:s_end]\n        seq_y_mark = self.data_stamp[r_begin:r_end]\n\n        return seq_x, seq_y, seq_x_mark, seq_y_mark\n\n    def __len__(self):\n        return len(self.data_x) - self.seq_len - self.pred_len + 1\n\n    def inverse_transform(self, data):\n        return self.scaler.inverse_transform(data)\n\n\nclass Dataset_Custom(Dataset):\n    def __init__(self, root_path, flag='train', size=None,\n                 features='S', data_path='ETTh1.csv',\n                 target='OT', scale=True, timeenc=0, freq='h'):\n        if size == None:\n            self.seq_len = 24 * 4 * 4\n            self.label_len = 24 * 4\n            self.pred_len = 24 * 4\n        else:\n            self.seq_len = size[0]\n            self.label_len = size[1]\n            self.pred_len = size[2]\n        assert flag in ['train', 'test', 'val']\n        type_map = {'train': 0, 'val': 1, 'test': 2}\n        self.set_type = type_map[flag]\n\n        self.features = features\n        self.target = target\n        self.scale = scale\n        self.timeenc = timeenc\n        self.freq = freq\n\n        self.root_path = root_path\n        self.data_path = data_path\n        self.__read_data__()\n\n    def __read_data__(self):\n        self.scaler = StandardScaler()\n        df_raw = pd.read_csv(os.path.join(self.root_path,\n                                          self.data_path))\n        df_raw = _prepare_custom_dataframe(df_raw, self.target, self.features)\n        num_train = int(len(df_raw) * 0.7)\n        num_test = int(len(df_raw) * 0.2)\n        num_vali = len(df_raw) - num_train - num_test\n        border1s = [0, num_train - self.seq_len, len(df_raw) - num_test - self.seq_len]\n        border2s = [num_train, num_train + num_vali, len(df_raw)]\n        border1 = border1s[self.set_type]\n        border2 = border2s[self.set_type]\n\n        if self.features == 'M' or self.features == 'MS':\n            cols_data = df_raw.columns[1:]\n            df_data = df_raw[cols_data]\n        elif self.features == 'S':\n            df_data = df_raw[[self.target]]\n\n        if self.scale:\n            train_data = df_data[border1s[0]:border2s[0]]\n            self.scaler.fit(train_data.values)\n            data = self.scaler.transform(df_data.values)\n        else:\n            data = df_data.values\n\n        df_stamp = df_raw[['date']][border1:border2]\n        df_stamp['date'] = pd.to_datetime(df_stamp.date)\n        if self.timeenc == 0:\n            df_stamp['month'] = df_stamp.date.apply(lambda row: row.month, 1)\n            df_stamp['day'] = df_stamp.date.apply(lambda row: row.day, 1)\n            df_stamp['weekday'] = df_stamp.date.apply(lambda row: row.weekday(), 1)\n            df_stamp['hour'] = df_stamp.date.apply(lambda row: row.hour, 1)\n            data_stamp = df_stamp.drop(['date'], 1).values\n        elif self.timeenc == 1:\n            data_stamp = time_features(pd.to_datetime(df_stamp['date'].values), freq=self.freq)\n            data_stamp = data_stamp.transpose(1, 0)\n\n        self.data_x = data[border1:border2]\n        self.data_y = data[border1:border2]\n        self.data_stamp = data_stamp\n\n    def __getitem__(self, index):\n        s_begin = index\n        s_end = s_begin + self.seq_len\n        r_begin = s_end - self.label_len\n        r_end = r_begin + self.label_len + self.pred_len\n\n        seq_x = self.data_x[s_begin:s_end]\n        seq_y = self.data_y[r_begin:r_end]\n        seq_x_mark = self.data_stamp[s_begin:s_end]\n        seq_y_mark = self.data_stamp[r_begin:r_end]\n\n        return seq_x, seq_y, seq_x_mark, seq_y_mark\n\n    def __len__(self):\n        return len(self.data_x) - self.seq_len - self.pred_len + 1\n\n    def inverse_transform(self, data):\n        return self.scaler.inverse_transform(data)\n\n\nclass Dataset_PEMS(Dataset):\n    def __init__(self, root_path, flag='train', size=None,\n                 features='S', data_path='ETTh1.csv',\n                 target='OT', scale=True, timeenc=0, freq='h', seasonal_patterns=None):\n        self.seq_len = size[0]\n        self.label_len = size[1]\n        self.pred_len = size[2]\n        assert flag in ['train', 'test', 'val']\n        type_map = {'train': 0, 'val': 1, 'test': 2}\n        self.set_type = type_map[flag]\n\n        self.features = features\n        self.target = target\n        self.scale = scale\n        self.timeenc = timeenc\n        self.freq = freq\n\n        self.root_path = root_path\n        self.data_path = data_path\n        self.__read_data__()\n\n    def __read_data__(self):\n        self.scaler = StandardScaler()\n        data_file = os.path.join(self.root_path, self.data_path)\n        print('data file:', data_file)\n        data = np.load(data_file, allow_pickle=True)\n        data = data['data'][:, :, 0]\n\n        train_ratio = 0.6\n        valid_ratio = 0.2\n        train_data = data[:int(train_ratio * len(data))]\n        valid_data = data[int(train_ratio * len(data)):int((train_ratio + valid_ratio) * len(data))]\n        test_data = data[int((train_ratio + valid_ratio) * len(data)):]\n        total_data = [train_data, valid_data, test_data]\n        data = total_data[self.set_type]\n\n        if self.scale:\n            self.scaler.fit(data)\n            data = self.scaler.transform(data)\n\n        df = pd.DataFrame(data)\n        df = df.fillna(method='ffill', limit=len(df)).fillna(method='bfill', limit=len(df)).values\n\n        self.data_x = df\n        self.data_y = df\n\n    def __getitem__(self, index):\n        if self.set_type == 2:  \n            s_begin = index * 12\n        else:\n            s_begin = index\n        s_end = s_begin + self.seq_len\n        r_begin = s_end - self.label_len\n        r_end = r_begin + self.label_len + self.pred_len\n\n        seq_x = self.data_x[s_begin:s_end]\n        seq_y = self.data_y[r_begin:r_end]\n        seq_x_mark = torch.zeros((seq_x.shape[0], 1))\n        seq_y_mark = torch.zeros((seq_y.shape[0], 1))\n\n        return seq_x, seq_y, seq_x_mark, seq_y_mark\n\n    def __len__(self):\n        if self.set_type == 2: \n            return (len(self.data_x) - self.seq_len - self.pred_len + 1) // 12\n        else:\n            return len(self.data_x) - self.seq_len - self.pred_len + 1\n\n    def inverse_transform(self, data):\n        return self.scaler.inverse_transform(data)\n\n\nclass Dataset_Solar(Dataset):\n    def __init__(self, root_path, flag='train', size=None,\n                 features='S', data_path='ETTh1.csv',\n                 target='OT', scale=True, timeenc=0, freq='h'):\n        self.seq_len = size[0]\n        self.label_len = size[1]\n        self.pred_len = size[2]\n        assert flag in ['train', 'test', 'val']\n        type_map = {'train': 0, 'val': 1, 'test': 2}\n        self.set_type = type_map[flag]\n\n        self.features = features\n        self.target = target\n        self.scale = scale\n        self.timeenc = timeenc\n        self.freq = freq\n\n        self.root_path = root_path\n        self.data_path = data_path\n        self.__read_data__()\n\n    def __read_data__(self):\n        self.scaler = StandardScaler()\n        df_raw = []\n        with open(os.path.join(self.root_path, self.data_path), \"r\", encoding='utf-8') as f:\n            for line in f.readlines():\n                line = line.strip('\\n').split(',')\n                data_line = np.stack([float(i) for i in line])\n                df_raw.append(data_line)\n        df_raw = np.stack(df_raw, 0)\n        df_raw = pd.DataFrame(df_raw)\n\n        num_train = int(len(df_raw) * 0.7)\n        num_test = int(len(df_raw) * 0.2)\n        num_valid = int(len(df_raw) * 0.1)\n        border1s = [0, num_train - self.seq_len, len(df_raw) - num_test - self.seq_len]\n        border2s = [num_train, num_train + num_valid, len(df_raw)]\n        border1 = border1s[self.set_type]\n        border2 = border2s[self.set_type]\n\n        df_data = df_raw.values\n\n        if self.scale:\n            train_data = df_data[border1s[0]:border2s[0]]\n            self.scaler.fit(train_data)\n            data = self.scaler.transform(df_data)\n        else:\n            data = df_data\n\n        self.data_x = data[border1:border2]\n        self.data_y = data[border1:border2]\n\n    def __getitem__(self, index):\n        s_begin = index\n        s_end = s_begin + self.seq_len\n        r_begin = s_end - self.label_len\n        r_end = r_begin + self.label_len + self.pred_len\n\n        seq_x = self.data_x[s_begin:s_end]\n        seq_y = self.data_y[r_begin:r_end]\n        seq_x_mark = torch.zeros((seq_x.shape[0], 1))\n        seq_y_mark = torch.zeros((seq_x.shape[0], 1))\n\n        return seq_x, seq_y, seq_x_mark, seq_y_mark\n\n    def __len__(self):\n        return len(self.data_x) - self.seq_len - self.pred_len + 1\n\n    def inverse_transform(self, data):\n        return self.scaler.inverse_transform(data)\n\n\nclass Dataset_Pred(Dataset):\n    def __init__(self, root_path, flag='pred', size=None,\n                 features='S', data_path='ETTh1.csv',\n                 target='OT', scale=True, inverse=False, timeenc=0, freq='15min', cols=None):\n        if size == None:\n            self.seq_len = 24 * 4 * 4\n            self.label_len = 24 * 4\n            self.pred_len = 24 * 4\n        else:\n            self.seq_len = size[0]\n            self.label_len = size[1]\n            self.pred_len = size[2]\n        assert flag in ['pred']\n\n        self.features = features\n        self.target = target\n        self.scale = scale\n        self.inverse = inverse\n        self.timeenc = timeenc\n        self.freq = freq\n        self.cols = cols\n        self.root_path = root_path\n        self.data_path = data_path\n        self.__read_data__()\n\n    def __read_data__(self):\n        self.scaler = StandardScaler()\n        df_raw = pd.read_csv(os.path.join(self.root_path,\n                                          self.data_path))\n        df_raw = _prepare_custom_dataframe(df_raw, self.target, self.features)\n        if self.cols:\n            keep_cols = ['date']\n            keep_cols.extend([col for col in self.cols if col in df_raw.columns and col != 'date'])\n            if self.target in df_raw.columns and self.target not in keep_cols:\n                keep_cols.append(self.target)\n            df_raw = df_raw[keep_cols]\n        border1 = len(df_raw) - self.seq_len\n        border2 = len(df_raw)\n\n        if self.features == 'M' or self.features == 'MS':\n            cols_data = df_raw.columns[1:]\n            df_data = df_raw[cols_data]\n        elif self.features == 'S':\n            df_data = df_raw[[self.target]]\n\n        if self.scale:\n            self.scaler.fit(df_data.values)\n            data = self.scaler.transform(df_data.values)\n        else:\n            data = df_data.values\n\n        tmp_stamp = df_raw[['date']][border1:border2]\n        tmp_stamp['date'] = pd.to_datetime(tmp_stamp.date)\n        pred_dates = pd.date_range(tmp_stamp.date.values[-1], periods=self.pred_len + 1, freq=self.freq)\n\n        df_stamp = pd.DataFrame(columns=['date'])\n        df_stamp.date = list(tmp_stamp.date.values) + list(pred_dates[1:])\n        if self.timeenc == 0:\n            df_stamp['month'] = df_stamp.date.apply(lambda row: row.month, 1)\n            df_stamp['day'] = df_stamp.date.apply(lambda row: row.day, 1)\n            df_stamp['weekday'] = df_stamp.date.apply(lambda row: row.weekday(), 1)\n            df_stamp['hour'] = df_stamp.date.apply(lambda row: row.hour, 1)\n            df_stamp['minute'] = df_stamp.date.apply(lambda row: row.minute, 1)\n            df_stamp['minute'] = df_stamp.minute.map(lambda x: x // 15)\n            data_stamp = df_stamp.drop(['date'], 1).values\n        elif self.timeenc == 1:\n            data_stamp = time_features(pd.to_datetime(df_stamp['date'].values), freq=self.freq)\n            data_stamp = data_stamp.transpose(1, 0)\n\n        self.data_x = data[border1:border2]\n        if self.inverse:\n            self.data_y = df_data.values[border1:border2]\n        else:\n            self.data_y = data[border1:border2]\n        self.data_stamp = data_stamp\n\n    def __getitem__(self, index):\n        s_begin = index\n        s_end = s_begin + self.seq_len\n        r_begin = s_end - self.label_len\n        r_end = r_begin + self.label_len + self.pred_len\n\n        seq_x = self.data_x[s_begin:s_end]\n        if self.inverse:\n            seq_y = self.data_x[r_begin:r_begin + self.label_len]\n        else:\n            seq_y = self.data_y[r_begin:r_begin + self.label_len]\n        seq_x_mark = self.data_stamp[s_begin:s_end]\n        seq_y_mark = self.data_stamp[r_begin:r_end]\n\n        return seq_x, seq_y, seq_x_mark, seq_y_mark\n\n    def __len__(self):\n        return len(self.data_x) - self.seq_len + 1\n\n    def inverse_transform(self, data):\n        return self.scaler.inverse_transform(data)\n", "utils/metrics.py": "import numpy as np\n\n\nEPS = 1e-8\n\n\ndef RSE(pred, true):\n    numerator = np.sqrt(np.sum((true - pred) ** 2))\n    denominator = np.sqrt(np.sum((true - true.mean()) ** 2))\n    return numerator / (denominator + EPS)\n\n\ndef CORR(pred, true):\n    u = ((true - true.mean(0)) * (pred - pred.mean(0))).sum(0)\n    d = np.sqrt(((true - true.mean(0)) ** 2 * (pred - pred.mean(0)) ** 2).sum(0))\n    return (u / (d + EPS)).mean(-1)\n\n\ndef MAE(pred, true):\n    return np.mean(np.abs(pred - true))\n\n\ndef MSE(pred, true):\n    return np.mean((pred - true) ** 2)\n\n\ndef RMSE(pred, true):\n    return np.sqrt(MSE(pred, true))\n\n# Troubleshooting for PEMS Nov 8\n# def MAPE(pred, true):\n#     return np.mean(np.abs((pred - true) / true))\ndef MAPE(pred, true):\n    mape = np.abs((pred - true) / (true + EPS))\n    mape = np.where(mape > 5, 0, mape)\n    return np.mean(mape)\n\n\ndef MSPE(pred, true):\n    return np.mean(np.square((pred - true) / (true + EPS)))\n\n\ndef SMAPE(pred, true):\n    denominator = np.abs(pred) + np.abs(true) + EPS\n    return np.mean(2.0 * np.abs(pred - true) / denominator)\n\n\ndef WAPE(pred, true):\n    return np.sum(np.abs(pred - true)) / (np.sum(np.abs(true)) + EPS)\n\n\ndef R2(pred, true):\n    ss_res = np.sum((true - pred) ** 2)\n    ss_tot = np.sum((true - np.mean(true)) ** 2)\n    return 1.0 - (ss_res / (ss_tot + EPS))\n\n\ndef metric(pred, true):\n    mae = MAE(pred, true)\n    mse = MSE(pred, true)\n    rmse = RMSE(pred, true)\n    mape = MAPE(pred, true)\n    mspe = MSPE(pred, true)\n\n    return mae, mse, rmse, mape, mspe\n\n\ndef metric_extended(pred, true):\n    mae, mse, rmse, mape, mspe = metric(pred, true)\n    rse = RSE(pred, true)\n    corr = CORR(pred, true)\n    smape = SMAPE(pred, true)\n    wape = WAPE(pred, true)\n    r2 = R2(pred, true)\n\n    if isinstance(corr, np.ndarray):\n        corr = float(np.mean(corr))\n\n    return {\n        'mae': float(mae),\n        'mse': float(mse),\n        'rmse': float(rmse),\n        'mape': float(mape),\n        'mspe': float(mspe),\n        'rse': float(rse),\n        'corr': float(corr),\n        'smape': float(smape),\n        'wape': float(wape),\n        'r2': float(r2),\n    }\n", "experiments/exp_long_term_forecasting.py": "from torch.optim import lr_scheduler\n\nfrom data_provider.data_factory import data_provider\nfrom experiments.exp_basic import Exp_Basic\nfrom utils.tools import EarlyStopping, adjust_learning_rate, visual\nfrom utils.metrics import metric, metric_extended\nimport torch\nimport torch.nn as nn\nfrom torch import optim\nimport os\nimport time\nimport warnings\nimport numpy as np\n\nwarnings.filterwarnings('ignore')\n\ntorch.autograd.set_detect_anomaly(True)\n\n\nclass Exp_Long_Term_Forecast(Exp_Basic):\n    def __init__(self, args):\n        super(Exp_Long_Term_Forecast, self).__init__(args)\n\n    def _build_model(self):\n        model = self.model_dict[self.args.model].Model(self.args).float()\n\n        if self.args.use_multi_gpu and self.args.use_gpu:\n            model = nn.DataParallel(model, device_ids=self.args.device_ids)        #wrapper for using multiple gpu\n        return model\n\n    def _get_data(self, flag):\n        data_set, data_loader = data_provider(self.args, flag)\n        return data_set, data_loader\n\n    def _select_optimizer(self):\n        model_optim = optim.AdamW(self.model.parameters(), lr=self.args.learning_rate)\n        return model_optim\n\n    def _select_criterion(self):\n        if self.args.data == 'PEMS':\n            criterion = nn.L1Loss()  #Mean Absolute Error\n        else:\n            criterion = nn.MSELoss() #Mean Squared Error\n        return criterion\n\n    def vali(self, vali_data, vali_loader, criterion):\n        total_loss = []\n        self.model.eval()\n        with torch.no_grad():    #Disables gradient computation.\n            for i, (batch_x, batch_y, batch_x_mark, batch_y_mark) in enumerate(vali_loader):\n                batch_x = batch_x.float().to(self.device)\n                batch_y = batch_y.float()\n\n                if 'PEMS' in self.args.data or 'Solar' in self.args.data:\n                    batch_x_mark = None\n                    batch_y_mark = None\n                else:\n                    batch_x_mark = batch_x_mark.float().to(self.device)\n                    batch_y_mark = batch_y_mark.float().to(self.device)\n\n                dec_inp = torch.zeros_like(batch_y[:, -self.args.pred_len:, :]).float() #creates a tensor of zeros that has the same shape as the future part of batch_y.\n                dec_inp = torch.cat([batch_y[:, :self.args.label_len, :], dec_inp], dim=1).float().to(self.device) #cat is concatinate\n\n                if self.args.use_amp:        #Automatic Mixed Precision (AMP) is a technique in deep learning that uses both 32-bit and 16-bit floating point numbers during training to make training faster and use less GPU memory, while keeping model accuracy.\n                    with torch.cuda.amp.autocast():\n                        if self.args.output_attention:\n                            outputs = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n                        else:\n                            outputs, _ = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n                else:\n                    if self.args.output_attention:\n                        outputs = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n                    else:\n                        outputs, _ = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n                f_dim = -1 if self.args.features == 'MS' else 0\n                outputs = outputs[:, -self.args.pred_len:, f_dim:] # in this  --self.args.pred_len is done only for safty\n                batch_y = batch_y[:, -self.args.pred_len:, f_dim:].to(self.device)\n\n                pred = outputs.detach().cpu() #detach() is used to remove the tensor from the computation graph.\n                true = batch_y.detach().cpu() #cpu() is used to move the tensor from the GPU to the CPU.\n\n                if self.args.data == 'PEMS':\n                    B, T, C = pred.shape\n                    pred = pred.numpy()\n                    true = true.numpy()\n                    pred = vali_data.inverse_transform(pred.reshape(-1, C)).reshape(B, T, C)\n                    true = vali_data.inverse_transform(true.reshape(-1, C)).reshape(B, T, C)\n                    mae, mse, rmse, mape, mspe = metric(pred, true)\n                    total_loss.append(mae)\n                else:\n                    loss = criterion(pred, true)\n                    total_loss.append(loss)\n\n        total_loss = np.average(total_loss)\n        self.model.train()\n        return total_loss\n\n    def train(self, setting):      #start again from here\n        train_data, train_loader = self._get_data(flag='train')\n        vali_data, vali_loader = self._get_data(flag='val')\n        test_data, test_loader = self._get_data(flag='test')\n\n        path = os.path.join(self.args.checkpoints, setting)\n        if not os.path.exists(path):\n            os.makedirs(path)\n\n        time_now = time.time()\n\n        train_steps = len(train_loader)\n        early_stopping = EarlyStopping(patience=self.args.patience, verbose=True)\n\n        model_optim = self._select_optimizer()\n        criterion = self._select_criterion()\n\n        if self.args.lradj == 'TST':\n            scheduler = lr_scheduler.OneCycleLR(optimizer=model_optim,\n                                                steps_per_epoch=train_steps,\n                                                pct_start=self.args.pct_start,\n                                                epochs=self.args.train_epochs,\n                                                max_lr=self.args.learning_rate)\n\n\n        if self.args.use_amp:\n            scaler = torch.cuda.amp.GradScaler()\n        # # Efficiency: dynamic memory footprint\n        # # Track dynamic memory usage over an epoch\n        # torch.cuda.reset_peak_memory_stats()  # Reset peak memory tracking\n\n        for epoch in range(self.args.train_epochs):\n            iter_count = 0\n            train_loss = []\n\n            self.model.train()\n            epoch_time = time.time()\n            for i, (batch_x, batch_y, batch_x_mark, batch_y_mark) in enumerate(train_loader):\n                iter_count += 1\n                model_optim.zero_grad()\n                batch_x = batch_x.float().to(self.device)\n\n                batch_y = batch_y.float().to(self.device)\n                if 'PEMS' in self.args.data or 'Solar' in self.args.data:\n                    batch_x_mark = None\n                    batch_y_mark = None\n                else:\n                    batch_x_mark = batch_x_mark.float().to(self.device)\n                    batch_y_mark = batch_y_mark.float().to(self.device)\n\n                dec_inp = torch.zeros_like(batch_y[:, -self.args.pred_len:, :]).float()\n                dec_inp = torch.cat([batch_y[:, :self.args.label_len, :], dec_inp], dim=1).float().to(self.device)\n\n                if self.args.use_amp:\n                    with torch.cuda.amp.autocast():\n                        if self.args.output_attention:\n                            outputs = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n                        else:\n                            outputs, _ = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n\n                        f_dim = -1 if self.args.features == 'MS' else 0\n                        outputs = outputs[:, -self.args.pred_len:, f_dim:]\n                        batch_y = batch_y[:, -self.args.pred_len:, f_dim:].to(self.device)\n                        loss = criterion(outputs, batch_y) \n                        train_loss.append(loss.item())\n                else:\n                    if self.args.output_attention:\n                        outputs = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n                    else:\n                        outputs, attn = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n\n                    f_dim = -1 if self.args.features == 'MS' else 0                        \n                    outputs = outputs[:, -self.args.pred_len:, f_dim:]\n                    batch_y = batch_y[:, -self.args.pred_len:, f_dim:].to(self.device)\n                    \n                    loss = criterion(outputs, batch_y) + self.args.l1_weight * attn[0] \n                    train_loss.append(loss.item())\n\n                if (i + 1) % 30 == 0:\n                    print(\"\\titers: {0}, epoch: {1} | loss: {2:.7f}\".format(i + 1, epoch + 1, loss.item()))\n                    speed = (time.time() - time_now) / iter_count\n                    left_time = speed * ((self.args.train_epochs - epoch) * train_steps - i)\n                    print('\\tspeed: {:.4f}s/iter; left time: {:.4f}s'.format(speed, left_time))\n                    iter_count = 0\n                    time_now = time.time()\n\n                if self.args.use_amp:\n                    scaler.scale(loss).backward()\n                    scaler.step(model_optim)\n                    scaler.update()\n                else:\n                    loss.backward()\n                    model_optim.step()\n                    # # Efficiency: dynamic memory footprint\n                    # # Record current and peak memory usage after processing this batch\n                    # current_memory = torch.cuda.memory_allocated()\n                    # peak_memory = torch.cuda.max_memory_allocated()\n                    # print(f\"Current memory: {current_memory / (1024 ** 2):.2f} MB, Peak memory: {peak_memory / (1024 ** 2):.2f} MB\")\n\n                if self.args.lradj == 'TST':\n                    adjust_learning_rate(model_optim, epoch + 1, self.args, scheduler, printout=False)\n                    scheduler.step()\n\n\n            print(\"Epoch: {} cost time: {}\".format(epoch + 1, time.time() - epoch_time))\n            train_loss = np.average(train_loss)\n            vali_loss = self.vali(vali_data, vali_loader, criterion)\n            test_loss = self.vali(test_data, test_loader, criterion)\n\n            print(\"Epoch: {0}, Steps: {1} | Train Loss: {2:.7f} Vali Loss: {3:.7f} Test Loss: {4:.7f}\".format(\n                epoch + 1, train_steps, train_loss, vali_loss, test_loss))\n            early_stopping(vali_loss, self.model, path)\n            if early_stopping.early_stop:\n                print(\"Early stopping\")\n                break\n\n            if self.args.lradj != 'TST':\n                adjust_learning_rate(model_optim, epoch + 1, self.args)\n            else:\n                adjust_learning_rate(model_optim, epoch + 1, self.args, scheduler)\n\n        \n        best_model_path = path + '/' + 'checkpoint.pth'\n        self.model.load_state_dict(torch.load(best_model_path))\n\n        return self.model\n\n    def test(self, setting, test=0):\n        test_data, test_loader = self._get_data(flag='test')\n        if test:\n            print('loading model')\n            self.model.load_state_dict(torch.load(os.path.join('./checkpoints/' + setting, 'checkpoint.pth')))\n\n        preds = []\n        trues = []\n        folder_path = './checkpoints/' + setting + '/'\n        if not os.path.exists(folder_path):\n            os.makedirs(folder_path)\n\n        self.model.eval()\n        with torch.no_grad():\n            for i, (batch_x, batch_y, batch_x_mark, batch_y_mark) in enumerate(test_loader):\n                batch_x = batch_x.float().to(self.device)\n                batch_y = batch_y.float().to(self.device)\n\n                if 'PEMS' in self.args.data or 'Solar' in self.args.data:\n                    batch_x_mark = None\n                    batch_y_mark = None\n                else:\n                    batch_x_mark = batch_x_mark.float().to(self.device)\n                    batch_y_mark = batch_y_mark.float().to(self.device)\n\n\n                dec_inp = torch.zeros_like(batch_y[:, -self.args.pred_len:, :]).float()\n                dec_inp = torch.cat([batch_y[:, :self.args.label_len, :], dec_inp], dim=1).float().to(self.device)\n                # encoder - decoder\n                if self.args.use_amp:\n                    with torch.cuda.amp.autocast():\n                        if self.args.output_attention:\n                            outputs = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n                        else:\n                            outputs, _ = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n                else:\n                    if self.args.output_attention:\n                        outputs = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n                    else:\n                        outputs, _ = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n\n                f_dim = -1 if self.args.features == 'MS' else 0\n                outputs = outputs[:, -self.args.pred_len:, f_dim:]\n                batch_y = batch_y[:, -self.args.pred_len:, f_dim:].to(self.device)\n                outputs = outputs.detach().cpu().numpy()\n                batch_y = batch_y.detach().cpu().numpy()\n\n                pred = outputs\n                true = batch_y\n\n                preds.append(pred)\n                trues.append(true)\n                if i % 20 == 0:\n                    input = batch_x.detach().cpu().numpy()\n                    if test_data.scale and self.args.inverse:\n                        shape = input.shape\n                        input = test_data.inverse_transform(input.squeeze(0)).reshape(shape)\n                    gt = np.concatenate((input[0, :, -1], true[0, :, -1]), axis=0)\n                    pd = np.concatenate((input[0, :, -1], pred[0, :, -1]), axis=0)\n                    visual(gt, pd, os.path.join(folder_path, str(i) + '.pdf'))\n\n        preds = np.array(preds)\n        trues = np.array(trues)\n        print('test shape:', preds.shape, trues.shape)\n        preds = preds.reshape(-1, preds.shape[-2], preds.shape[-1])\n        trues = trues.reshape(-1, trues.shape[-2], trues.shape[-1])\n        print('test shape:', preds.shape, trues.shape)\n\n        if self.args.data == 'PEMS':\n            B, T, C = preds.shape\n            preds = test_data.inverse_transform(preds.reshape(-1, C)).reshape(B, T, C)\n            trues = test_data.inverse_transform(trues.reshape(-1, C)).reshape(B, T, C)\n\n        # result save\n        folder_path = './checkpoints/' + setting + '/'\n        if not os.path.exists(folder_path):\n            os.makedirs(folder_path)\n\n        metrics = metric_extended(preds, trues)\n        print('mse:{}, mae:{}'.format(metrics['mse'], metrics['mae']))\n        print('rmse:{}, mape:{}, mspe:{}'.format(metrics['rmse'], metrics['mape'], metrics['mspe']))\n        print(\n            'rse:{}, corr:{}, smape:{}, wape:{}, r2:{}'.format(\n                metrics['rse'],\n                metrics['corr'],\n                metrics['smape'],\n                metrics['wape'],\n                metrics['r2'],\n            )\n        )\n        f = open(\"result_long_term_forecast.txt\", 'a')\n        f.write(setting + \"  \\n\")\n        metric_parts = [f'{name}:{value}' for name, value in metrics.items()]\n        f.write(', '.join(metric_parts))\n        f.write('\\n')\n        f.write('\\n')\n        f.close()\n\n    \n        return\n\n\n    def predict(self, setting, load=False):\n        pred_data, pred_loader = self._get_data(flag='pred')\n\n        if load:\n            path = os.path.join(self.args.checkpoints, setting)\n            best_model_path = path + '/' + 'checkpoint.pth'\n            self.model.load_state_dict(torch.load(best_model_path))\n\n        preds = []\n\n        self.model.eval()\n        with torch.no_grad():\n            for i, (batch_x, batch_y, batch_x_mark, batch_y_mark) in enumerate(pred_loader):\n                batch_x = batch_x.float().to(self.device)\n                batch_y = batch_y.float()\n                batch_x_mark = batch_x_mark.float().to(self.device)\n                batch_y_mark = batch_y_mark.float().to(self.device)\n\n    \n                dec_inp = torch.zeros_like(batch_y[:, -self.args.pred_len:, :]).float()\n                dec_inp = torch.cat([batch_y[:, :self.args.label_len, :], dec_inp], dim=1).float().to(self.device)\n                # encoder - decoder\n                if self.args.use_amp:\n                    with torch.cuda.amp.autocast():\n                        if self.args.output_attention:\n                            outputs = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n                        else:\n                            outputs, _ = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n                else:\n                    if self.args.output_attention:\n                        outputs = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n                    else:\n                        outputs, _ = self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)\n                outputs = outputs.detach().cpu().numpy()\n                if pred_data.scale and self.args.inverse:\n                    shape = outputs.shape\n                    outputs = pred_data.inverse_transform(outputs.squeeze(0)).reshape(shape)\n                preds.append(outputs)\n\n        preds = np.array(preds)\n        preds = preds.reshape(-1, preds.shape[-2], preds.shape[-1])\n\n        # result save\n        folder_path = './results/' + setting + '/'\n        if not os.path.exists(folder_path):\n            os.makedirs(folder_path)\n\n        np.save(folder_path + 'real_prediction.npy', preds)\n\n        return\n"}

for rel_path, content in FILES_TO_SYNC.items():
    abs_path = os.path.join(REPO_DIR, rel_path)
    os.makedirs(os.path.dirname(abs_path), exist_ok=True)
    with open(abs_path, 'w', encoding='utf-8', newline='') as f:
        f.write(content)
    print(f"Synced {rel_path}")

!sed -i 's/np\.Inf/np.inf/g' {REPO_DIR}/utils/tools.py
print("Repo sync complete.")


In [ ]:
FILE_ID = "1JOgmL2ntyxKg6eB132ytCbVZumiHnDLR"
OUTPUT_PATH = "/kaggle/working/dataset.zip"
DATASET_DIR = "./dataset"

if not os.path.exists(DATASET_DIR):
    !gdown https://drive.google.com/uc?id={FILE_ID} -O {OUTPUT_PATH}
    import zipfile
    os.makedirs(DATASET_DIR, exist_ok=True)
    with zipfile.ZipFile(OUTPUT_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)
    print("Dataset extracted successfully.")
else:
    print("Dataset already exists.")

for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:10]:
        print(f'{subindent}{file}')


In [ ]:
# smart_candidates = [
#     ("./dataset/SimpleTM_datasets/smartbuilding", "smart.csv"),
#     ("./dataset/SimpleTM_datasets/Smartbuilding", "smart.csv"),
#     (".", "smart.csv"),
# ]

# SMART_ROOT, SMART_PATH = None, None
# for root, path in smart_candidates:
#     candidate = os.path.join(root, path)
#     if os.path.exists(candidate):
#         SMART_ROOT, SMART_PATH = root, path
#         break

# if SMART_ROOT is None:
#     raise FileNotFoundError("smart.csv not found in the extracted dataset or repo root.")

# smart_df = pd.read_csv(os.path.join(SMART_ROOT, SMART_PATH))
# print("smart.csv shape:", smart_df.shape)
# print("smart.csv columns:")
# for i, col in enumerate(smart_df.columns, start=1):
#     print(f"{i:02d}. {col}")
# print() 
# print(smart_df.head(3).to_string())


In [ ]:
SELECTED_DATASETS = None
# Example: SELECTED_DATASETS = ["ETTh1", "Solar", "smartbuilding"]

all_datasets = [
    {"name": "ETTh1", "data": "ETTh1", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTh1.csv", "enc_in": 7, "d_model": 32, "d_ff": 32, "wv": "db1", "m": 3, "alpha": 1.0, "learning_rate": 0.0001, "batch_size": 32, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "type1", "freq": "h"},
    {"name": "ETTh2", "data": "ETTh2", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTh2.csv", "enc_in": 7, "d_model": 32, "d_ff": 32, "wv": "db1", "m": 3, "alpha": 1.0, "learning_rate": 0.0001, "batch_size": 32, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "type1", "freq": "h"},
    {"name": "ETTm1", "data": "ETTm1", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTm1.csv", "enc_in": 7, "d_model": 32, "d_ff": 32, "wv": "db1", "m": 3, "alpha": 1.0, "learning_rate": 0.0001, "batch_size": 32, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "type1", "freq": "t"},
    {"name": "ETTm2", "data": "ETTm2", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTm2.csv", "enc_in": 7, "d_model": 32, "d_ff": 32, "wv": "db1", "m": 3, "alpha": 1.0, "learning_rate": 0.0001, "batch_size": 32, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "type1", "freq": "t"},
    {"name": "weather", "data": "custom", "root": "./dataset/SimpleTM_datasets/weather", "path": "weather.csv", "enc_in": 21, "d_model": 32, "d_ff": 32, "wv": "db1", "m": 3, "alpha": 1.0, "learning_rate": 0.0001, "batch_size": 32, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "type1", "freq": "h"},
    {"name": "electricity", "data": "custom", "root": "./dataset/SimpleTM_datasets/electricity", "path": "electricity.csv", "enc_in": 321, "d_model": 128, "d_ff": 128, "wv": "db1", "m": 3, "alpha": 1.0, "learning_rate": 0.0001, "batch_size": 32, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "type1", "freq": "h"},
    {"name": "traffic", "data": "custom", "root": "./dataset/SimpleTM_datasets/traffic", "path": "traffic.csv", "enc_in": 862, "d_model": 256, "d_ff": 256, "wv": "db1", "m": 3, "alpha": 1.0, "learning_rate": 0.0001, "batch_size": 8, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "type1", "freq": "h"},
    {"name": "Solar", "data": "Solar", "root": "./dataset/SimpleTM_datasets/Solar", "path": "solar_AL.txt", "enc_in": 137, "d_model": 128, "d_ff": 256, "wv": "db8", "m": 3, "alpha": 0.0, "learning_rate": 0.003, "batch_size": 128, "train_epochs": 10, "patience": 3, "use_norm": 0, "lradj": "TST", "freq": "h"},
    {"name": "PEMS03", "data": "PEMS", "root": "./dataset/SimpleTM_datasets/PEMS", "path": "PEMS03.npz", "enc_in": 358, "d_model": 256, "d_ff": 1024, "wv": "bior3.1", "m": 3, "alpha": 0.1, "learning_rate": 0.002, "batch_size": 16, "train_epochs": 20, "patience": 10, "use_norm": 0, "lradj": "TST", "freq": "h"},
    {"name": "PEMS04", "data": "PEMS", "root": "./dataset/SimpleTM_datasets/PEMS", "path": "PEMS04.npz", "enc_in": 307, "d_model": 256, "d_ff": 1024, "wv": "bior3.1", "m": 3, "alpha": 0.1, "learning_rate": 0.002, "batch_size": 16, "train_epochs": 20, "patience": 10, "use_norm": 0, "lradj": "TST", "freq": "h"},
    {"name": "PEMS07", "data": "PEMS", "root": "./dataset/SimpleTM_datasets/PEMS", "path": "PEMS07.npz", "enc_in": 883, "d_model": 256, "d_ff": 512, "wv": "db1", "m": 3, "alpha": 0.1, "learning_rate": 0.002, "batch_size": 8, "train_epochs": 20, "patience": 10, "use_norm": 0, "lradj": "TST", "freq": "h"},
    {"name": "PEMS08", "data": "PEMS", "root": "./dataset/SimpleTM_datasets/PEMS", "path": "PEMS08.npz", "enc_in": 170, "d_model": 256, "d_ff": 1024, "wv": "db12", "m": 3, "alpha": 0.0, "learning_rate": 0.001, "batch_size": 16, "train_epochs": 20, "patience": 10, "use_norm": 0, "lradj": "TST", "freq": "h"},
    {"name": "smartbuilding", "data": "custom", "root": "./dataset/SimpleTM_datasets/smartbuilding", "path": "smart.csv", "enc_in": 37, "d_model": 64, "d_ff": 128, "wv": "db4", "m": 3, "alpha": 1.0, "learning_rate": 0.0001, "batch_size": 32, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "type1", "freq": "h", "target": "Floor_Total(kW)"},
    # {"name": "smartbuilding", "data": "custom", "root": SMART_ROOT, "path": SMART_PATH, "enc_in": 37, "d_model": 64, "d_ff": 128, "wv": "db4", "m": 3, "alpha": 1.0, "learning_rate": 0.0001, "batch_size": 32, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "type1", "freq": "h", "target": "Floor_Total(kW)"},
]

experiments = [
    {"tag": "SWT_original", "model": "SimpleTM_SWT", "attention_mode": "original"},
    {"tag": "SWT_dual", "model": "SimpleTM_SWT", "attention_mode": "dual"},
    {"tag": "FFT_original", "model": "SimpleTM_FFT", "attention_mode": "original"},
    {"tag": "FFT_dual", "model": "SimpleTM_FFT", "attention_mode": "dual"},
]

if SELECTED_DATASETS is not None:
    datasets = [ds for ds in all_datasets if ds["name"] in SELECTED_DATASETS]
else:
    datasets = all_datasets

datasets = [ds for ds in datasets if os.path.exists(os.path.join(ds["root"], ds["path"]))]

OUTPUT_DIR = "/kaggle/working/SimpleTM_AllDatasets_Smart_Results"

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
if os.path.exists("./checkpoints"):
    shutil.rmtree("./checkpoints")
if os.path.exists("result_long_term_forecast.txt"):
    os.remove("result_long_term_forecast.txt")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/plots", exist_ok=True)

run_plan = pd.DataFrame(datasets)
run_plan.to_csv(os.path.join(OUTPUT_DIR, "run_plan.csv"), index=False)
print(run_plan[["name", "data", "path", "enc_in", "d_model", "d_ff"]].to_string(index=False))
print(f"\nDatasets: {len(datasets)}")
print(f"Variants per dataset: {len(experiments)}")
print(f"Total runs planned: {len(datasets) * len(experiments)}")


In [ ]:
for ds in datasets:
    print("\n" + "=" * 100)
    print(f"DATASET: {ds['name']} | variates={ds['enc_in']}")
    print("=" * 100 + "\n")

    for exp in experiments:
        print(f">>> Training {exp['tag']} on {ds['name']} <<<\n")
        unique_model_id = f"{ds['name']}_{exp['model']}_{exp['attention_mode']}"

        cmd = [
            "python", "-u", "run.py",
            "--is_training", "1",
            "--model", exp["model"],
            "--attention_mode", exp["attention_mode"],
            "--model_id", unique_model_id,
            "--data", ds["data"],
            "--root_path", ds["root"],
            "--data_path", ds["path"],
            "--features", "M",
            "--freq", ds["freq"],
            "--seq_len", "96",
            "--pred_len", "96",
            "--e_layers", "1",
            "--d_model", str(ds["d_model"]),
            "--d_ff", str(ds["d_ff"]),
            "--enc_in", str(ds["enc_in"]),
            "--dec_in", str(ds["enc_in"]),
            "--c_out", str(ds["enc_in"]),
            "--wv", ds["wv"],
            "--m", str(ds["m"]),
            "--alpha", str(ds["alpha"]),
            "--learning_rate", str(ds["learning_rate"]),
            "--batch_size", str(ds["batch_size"]),
            "--train_epochs", str(ds["train_epochs"]),
            "--patience", str(ds["patience"]),
            "--num_workers", "2",
            "--lradj", ds["lradj"],
            "--use_norm", str(ds["use_norm"]),
            "--checkpoints", f"{OUTPUT_DIR}/checkpoints/",
            "--fix_seed", "2025"
        ]

        if "target" in ds:
            cmd.extend(["--target", ds["target"]])

        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        for line in process.stdout:
            print(line, end="")
        process.wait()

        if process.returncode == 0:
            print(f"\nSUCCESS: Finished {exp['tag']} on {ds['name']}\n")
        else:
            raise RuntimeError(f"Run failed for {ds['name']} | {exp['tag']} with exit code {process.returncode}")

print("\nGathering generated PDF plots...")
if os.path.exists('./checkpoints'):
    for root_dir, dirs, files in os.walk('./checkpoints'):
        for file in files:
            if file.endswith('.pdf'):
                folder_name = os.path.basename(root_dir)
                source_path = os.path.join(root_dir, file)
                target_plot_dir = os.path.join(OUTPUT_DIR, 'plots', folder_name)
                os.makedirs(target_plot_dir, exist_ok=True)
                shutil.copy(source_path, os.path.join(target_plot_dir, file))
    print(f"Copied PDF plots to {OUTPUT_DIR}/plots/")

if os.path.exists('result_long_term_forecast.txt'):
    shutil.copy('result_long_term_forecast.txt', os.path.join(OUTPUT_DIR, 'result_long_term_forecast.txt'))
    print("Copied metric text results to output directory.")


In [ ]:
import re

results_file = os.path.join(OUTPUT_DIR, 'result_long_term_forecast.txt')
metric_order = ['MSE', 'MAE', 'RMSE', 'MAPE', 'MSPE', 'RSE', 'CORR', 'SMAPE', 'WAPE', 'R2']
pattern = re.compile(r'^(?P<dataset>[^_]+)_(?P<model>SimpleTM_(?:SWT|FFT))_(?P<attention>original|dual)_')
metric_pattern = re.compile(r'([a-zA-Z0-9_]+):([^,]+)')

if os.path.exists(results_file):
    with open(results_file, 'r', encoding='utf-8') as f:
        content = f.read().strip()

    entries = content.split('\n\n') if content else []
    rows = []
    for entry in entries:
        lines = [line.strip() for line in entry.splitlines() if line.strip()]
        if len(lines) < 2:
            continue
        setting = lines[0]
        metrics_line = lines[-1]
        match = pattern.search(setting)
        if not match:
            continue

        model_type = match.group('model')
        attention_mode = match.group('attention')
        tokenization = 'SWT' if model_type.endswith('SWT') else 'FFT'
        row = {
            'Dataset': match.group('dataset'),
            'Model': model_type,
            'AttentionMode': attention_mode,
            'Variant': f'{tokenization}_{attention_mode}',
            'Setting': setting,
        }
        for metric_name, value in metric_pattern.findall(metrics_line):
            try:
                row[metric_name.upper()] = float(value.strip())
            except ValueError:
                pass
        rows.append(row)

    if rows:
        df = pd.DataFrame(rows).sort_values(['Dataset', 'Model', 'AttentionMode']).reset_index(drop=True)
        print(df.to_string(index=False))

        raw_path = os.path.join(OUTPUT_DIR, 'raw_metrics.csv')
        df.to_csv(raw_path, index=False)

        value_columns = [col for col in metric_order if col in df.columns]
        pivot_df = df.pivot(index='Dataset', columns='Variant', values=value_columns)
        variant_order = ['SWT_original', 'SWT_dual', 'FFT_original', 'FFT_dual']
        pivot_df = pivot_df.reindex(columns=variant_order, level=1)
        final_path = os.path.join(OUTPUT_DIR, 'final_metrics.csv')
        pivot_df.to_csv(final_path)

        print(f'\nSaved raw results to: {raw_path}')
        print(f'Saved final table to: {final_path}')
    else:
        print('No valid metrics found in result file.')
else:
    print('Result file not found.')


In [ ]:
print("Packing logs, weights, plots, and results...")
archive_path = shutil.make_archive('/kaggle/working/SimpleTM_AllDatasets_Smart_Results', 'zip', OUTPUT_DIR)
print(f"\nDONE! Download {archive_path} from the Kaggle Output pane.")
